In [2]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [3]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


Querying owner: HTOC Org


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,dnsActive,whoisActive,legacyLink,tags.data,associatedGroups.data,ip,source,address,url,indicator
0,5629499557013355,2025-07-08T14:41:21Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-31T07:48:47Z,4.0,81.0,"On July 3, 2025 multiple emails were received ...",...,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 6755399452000619, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,click.smarterretirementsolutions.com
1,6755399447112815,2025-05-15T14:41:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-31T07:48:43Z,3.0,69.0,IRT 67143,...,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 32811, 'name': 'suspicious', 'lastUsed...","[{'id': 6755399448000094, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,congressionalaquarium.com
2,5629499554313220,2025-06-11T14:07:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-31T07:48:43Z,5.0,93.0,NaN,...,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 505635, 'name': 'DragonForce Ransomwar...","[{'id': 6755399449000723, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,com.ar
3,5272753,2025-01-31T11:46:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-31T07:48:43Z,2.0,5.0,Executive Summary\r\nInsikt Group has identifi...,...,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 461655, 'name': 'TA582', 'lastUsed': '...","[{'id': 6755399444000353, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,www.totem.tech
4,4584086,2024-04-29T17:54:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-31T07:48:43Z,3.0,66.0,An alert triggered for a backdoor that was pre...,...,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 495391, 'name': 'NIH_TAM', 'lastUsed':...","[{'id': 5629499541006432, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,autorun.inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1324,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84.0,NaN,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1326,4778127,2024-07-24T16:46:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-07-25T11:18:30Z,4.0,83.0,VA user reported suspicious External email to ...,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 358785, 'dateAdded': '2024-07-24T16:44...",NaN,https://epicphysicianstaffing.com,NaN,epicphysicianstaffing.com,epicphysicianstaffing.com
1327,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83.0,NaN,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,https://google,NaN,google,google
1328,4517160,2024-02-05T17:10:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-05T23:23:34Z,3.0,67.0,NaN,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...","[{'id': 313616, 'dateAdded': '2024-02-05T16:18...",NaN,https://realinvestmentadvice.com/,NaN,realinvestmentadvice.com/,realinvestmentadvice.com/


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_23000\3485874090.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250731.csv']
Loaded data from 1 files.


In [5]:
import pandas as pd

df = observed_src.copy()

df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

groups_exploded = (
    df[['indicator', 'associatedGroups.data']]
      .explode('associatedGroups.data')
      .dropna(subset=['associatedGroups.data'])
)

group_norm = pd.json_normalize(
    groups_exploded['associatedGroups.data']
).rename(columns={'id':'group_id','name':'group_name'})

groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

group_agg = (
    groups_exploded
      .drop_duplicates(subset=['indicator','group_id','group_name'])
      .groupby('indicator', as_index=False)
      .agg({
          'group_id':   lambda ids: list(ids),
          'group_name': lambda names: list(names)
      })
      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','hostName','dnsActive','whoisActive','source',
    'address','url'
]

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c not in ['address','ip','source','url','legacyLink']]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
for col in ['partners','group_ids','group_names'] + tag_fields:
    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners,group_ids,group_names
0,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-29T07:38:17Z,3.0,71.0,...,"2025-07-31T02:51:16Z, 2025-07-30T20:53:43Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
1,103.124.139.170,6755399460008265,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:38:25Z,4.0,70.0,...,"2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 20...",,,,,,,DHA,5629499544001057,INDICATOR NOTICE 25178.1 – Iran-Aligned Hackti...
2,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:55Z,3.0,73.0,...,"2025-07-31T02:51:16Z, 2025-07-30T20:53:43Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
3,103.186.30.186,5629499560265670,2025-07-21T13:35:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T19:09:03Z,5.0,89.0,...,"2025-07-25T18:04:04Z, 2025-07-23T16:44:02Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, NIH, IHS, HHS","6755399454000275, 5629499547000892, 5629499547...","SharePoint Zero-Day Exploitation Incident, HTO..."
4,103.203.59.0,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:52Z,3.0,70.0,...,"2025-07-31T02:51:16Z, 2025-07-30T20:53:43Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
872,scan.cypex.ai,6755399461914247,2025-07-09T14:26:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:00Z,3.0,77.0,...,"2025-07-31T00:42:25Z, 2025-07-31T02:49:08Z, 20...",,,,,,,"NIH, IHS",,
873,svgarchive.com,5263222,2025-01-22T18:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-30T16:33:10Z,0.0,0.0,...,"2025-05-30T16:59:39Z, 2025-07-23T17:48:15Z, 20...",,,,,,,NIH,"6755399448004173, 546474",NIH IRT - 65313 A FYSA Traffic Notification - ...
874,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:38:59Z,3.0,47.0,...,"2025-07-31T02:51:16Z, 2025-07-30T17:20:20Z, 20...",Adversaries may send phishing messages to gain...,T1566,['Initial Access'],1.0,"['Linux', 'macOS', 'Windows', 'SaaS', 'Identit...",6.0,"DHA, NIH, IHS",535913,NIH Phishing Emails 11/27/2024
875,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:04Z,4.0,50.0,...,"2025-07-31T02:51:16Z, 2025-07-30T20:53:43Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS","332131, 331131","HTOC-20240423-0923-A, VA Hosts Reaching Out to..."


In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

display(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        display(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        display("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    display(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


Enriching 877 indicators with Shodan & VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL aka.ms/learnaboutsenderidentification cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host bimbinlombardia.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host biologyinsights.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host brppharma.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress cameron@yourpensionmeeting.com cannot be enriched with Shodan because the indicator type isn\'t supported.",

Successfully enriched and merged 850 indicators.
27 indicators failed to enrich.


In [8]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-29T07:38:17Z,3.0,71.0,INC9067814,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,6.0
1,103.124.139.170,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:38:25Z,4.0,70.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Bandung],[Indonesia],[gmdp.net.id],[ptr-103-124-139-170.gmdp.net.id],[PT.Global Media Data Prima],"[{'transport': 'tcp', 'port': 8291, 'product':...",[PT.Global Media Data Prima],None,None,3.0
2,103.149.86.208,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:55Z,3.0,73.0,INC9097535,...,[Hanoi],[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],None,5.0
3,103.186.30.186,2025-07-21T13:35:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T19:09:03Z,5.0,89.0,"On July 18, security researchers and CISA disc...",...,[Bandung],[Indonesia],None,None,[CV Andhika Pratama Sanggoro],"[{'transport': 'tcp', 'port': 2000, 'product':...",[SMK Adi Sanggoro],None,None,10.0
4,103.203.59.0,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:52Z,3.0,70.0,RITM0581780/TASK0877793,...,[Tokyo],[Japan],[ipip.net],[scan-59-0.security.ipip.net],"[Beijing Tiantexin Tech. Co., Ltd.]","[{'transport': 'tcp', 'port': 22, 'product': '...","[Beijing Tiantexin Tech. Co., Ltd.]",[scanner],"[{'name': 'CVE-2008-3844', 'port': 22, 'descri...",3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
872,scan.cypex.ai,2025-07-09T14:26:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:00Z,3.0,77.0,INC9137833,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
873,svgarchive.com,2025-01-22T18:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-30T16:33:10Z,0.0,0.0,CSO received an email from CSIRC regarding a p...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
874,vtext.com,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:38:59Z,3.0,47.0,The correlation search is based on an automate...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
875,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:04Z,4.0,50.0,The Department of Veterans Affairs (VA) receiv...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def main():
    df_full = recent_tags
    today = pd.Timestamp(datetime.utcnow().date())
    dates = [today - pd.Timedelta(days=i) for i in range(7)]
    daily = [{'indicator': ind, 'date': d, 'count': np.random.poisson(5)} for ind in df_full['indicator'] for d in dates]
    daily_counts_df = pd.DataFrame(daily)
    source_sightings_df = pd.DataFrame([{'indicator': ind, 'ownerId': oid} for ind in df_full['indicator'] for oid in df_full['ownerId'].unique()])
    trust_map = {oid: np.random.choice([1, 5, 7, 10]) for oid in df_full['ownerId'].unique()}
    df_res = run_threat_pipeline(df_full, daily_counts_df, source_sightings_df, trust_map)
    # Instead of displaying each row, return the full dataframe for further use
    display(df_res)
    return df_res

Pulled 877 documents from enriched_observation_Data.


In [16]:
recent_tags

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,688b71d1f7ee26fe3d60e6f8,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-29T07:38:17Z,3.0,71.0,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,6.0
1,688b71d1f7ee26fe3d60e6f9,103.124.139.170,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T14:38:25Z,4.0,70.0,...,[Bandung],[Indonesia],[gmdp.net.id],[ptr-103-124-139-170.gmdp.net.id],[PT.Global Media Data Prima],"[{'transport': 'tcp', 'port': 8291, 'product':...",[PT.Global Media Data Prima],None,None,3.0
2,688b71d1f7ee26fe3d60e6fa,103.149.86.208,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:55Z,3.0,73.0,...,[Hanoi],[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],None,5.0
3,688b71d1f7ee26fe3d60e6fb,103.186.30.186,2025-07-21T13:35:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T19:09:03Z,5.0,89.0,...,[Bandung],[Indonesia],None,None,[CV Andhika Pratama Sanggoro],"[{'transport': 'tcp', 'port': 2000, 'product':...",[SMK Adi Sanggoro],None,None,10.0
4,688b71d1f7ee26fe3d60e6fc,103.203.59.0,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-30T07:35:52Z,3.0,70.0,...,[Tokyo],[Japan],[ipip.net],[scan-59-0.security.ipip.net],"[Beijing Tiantexin Tech. Co., Ltd.]","[{'transport': 'tcp', 'port': 22, 'product': '...","[Beijing Tiantexin Tech. Co., Ltd.]",[scanner],"[{'name': 'CVE-2008-3844', 'port': 22, 'descri...",3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
872,688b71d1f7ee26fe3d60ea60,scan.cypex.ai,2025-07-09T14:26:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:00Z,3.0,77.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
873,688b71d1f7ee26fe3d60ea61,svgarchive.com,2025-01-22T18:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-30T16:33:10Z,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
874,688b71d1f7ee26fe3d60ea62,vtext.com,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-29T07:38:59Z,3.0,47.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
875,688b71d1f7ee26fe3d60ea63,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-30T07:36:04Z,4.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [133]:
recent_tags[recent_tags['indicator'] == '104.21.48.1']

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
21,688b71d1f7ee26fe3d60e70d,104.21.48.1,2025-01-08T20:02:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-30T17:04:39Z,0.0,0.0,...,[San Francisco],[United States],"[sbg.life, surveillancecameraplayers.org]","[sbg.life, surveillancecameraplayers.org]","[Cloudflare, Inc.]","[{'transport': 'tcp', 'port': 80, 'product': '...","[Cloudflare, Inc.]",[cdn],None,4.0


In [128]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler

# --- 1. Temporal & Volume Features ---
def compute_temporal_volume_features(df, daily_counts_df, lambda_decay=0.1, today=None):
    if today is None:
        today = pd.Timestamp(datetime.utcnow().date())
    df = df.copy()
    df['dateAdded'] = pd.to_datetime(df['dateAdded']).dt.tz_localize(None).dt.normalize()
    df['lastObserved'] = pd.to_datetime(df['lastObserved']).dt.tz_localize(None).dt.normalize()
    df['raw_obs'] = df['observations']
    df['days_since_first_seen'] = (today - df['dateAdded']).dt.days
    df['decayed_obs'] = df['raw_obs'] * np.exp(-lambda_decay * df['days_since_first_seen'])
    df['log_decayed_obs'] = np.log1p(df['decayed_obs'])
    clip_val = df['log_decayed_obs'].quantile(0.95)
    df['log_decayed_obs'] = df['log_decayed_obs'].clip(upper=clip_val)
    def trend_burst(ind):
        counts = daily_counts_df.loc[daily_counts_df['indicator']==ind].sort_values('date')['count'].values
        if len(counts) >= 2:
            x = np.arange(len(counts))
            slope = np.polyfit(x, counts, 1)[0]
            burst = np.std(counts) / counts.mean() if counts.mean() > 0 else 0.0
        else:
            slope, burst = 0.0, 0.0
        return slope, burst
    tb = df['indicator'].apply(trend_burst)
    df[['obs_trend_slope','obs_burstiness']] = pd.DataFrame(tb.tolist(), index=df.index)
    return df

# --- 2. Source Diversity & Trust ---
def compute_source_diversity(df, source_sightings_df, trust_map):
    df = df.copy()
    df['num_distinct_sources'] = (
        source_sightings_df.groupby('indicator')['ownerId']
            .nunique()
            .reindex(df['indicator']).fillna(0).astype(int).values
    )
    agg = source_sightings_df.groupby('indicator')['ownerId'].apply(list)
    sums, maxs, ratios = [], [], []
    for ind in df['indicator']:
        owners = agg.get(ind, [])
        weights = [trust_map.get(o,5) for o in owners]
        sums.append(sum(weights))
        maxs.append(max(weights) if weights else 0)
        high = sum(w>=7 for w in weights)
        low = sum(w<7 for w in weights)
        ratios.append(high/(low if low>0 else 1))
    df['sum_source_weight'] = sums
    df['max_source_weight'] = maxs
    df['high_low_trust_ratio'] = ratios
    return df

# --- 3. False-Positive Signals ---
def compute_fp_signals(df):
    df = df.copy()
    df['fp_count'] = df.get('falsePositiveCount', 0)
    df['fp_rate'] = df['fp_count'] / df['observations'].replace(0, 1)
    return df

# --- 4. Contextual Metadata ---
def compute_context_metadata(df):
    df = df.copy()
    df = pd.concat([df, pd.get_dummies(df['type'], prefix='type')], axis=1)
    df['port_diversity'] = df['enrich_openPorts'].fillna('').apply(lambda p: len(set(str(p).split(','))) if p else 0)
    df['vt_malicious_count'] = df.get('enrich_vtMaliciousCount', 0)
    return df

# --- 5. Behavioral Signals ---
def compute_behavioral(df):
    df = df.copy()
    df['port_scan_confidence'] = df['enrich_tags'].str.contains('scanner', case=False).fillna(False).astype(int)
    return df

# --- 6. Derived Aggregates ---
def compute_derived(df, incidents_df=None, cal_scores=None):
    df = df.copy()
    df['ever_in_incident'] = incidents_df is not None and df['indicator'].isin(incidents_df['indicator'])
    df['peer_consensus_score'] = df.get('CAL_Score', cal_scores if cal_scores is not None else 0)
    return df

# --- 7. Unsupervised Confidence via LOF ---
def compute_confidence_score(df, feature_cols, lof_params=None):
    df = df.copy()
    X = df[feature_cols].fillna(0).values
    n_samples = X.shape[0]
    params = lof_params.copy() if lof_params else {}
    default_n = params.get('n_neighbors', 20)
    n_neighbors = min(default_n, max(n_samples - 1, 1))
    contamination = params.get('contamination', 0.1)
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    lof.fit(X)
    neg_factor = lof.negative_outlier_factor_
    anomaly_scores = -neg_factor
    if np.allclose(anomaly_scores, anomaly_scores[0]):
        df['conf_score'] = 0.5
    else:
        scaler = MinMaxScaler()
        df['conf_score'] = scaler.fit_transform(anomaly_scores.reshape(-1,1)).flatten()
    return df

# --- 8. Threat Scoring with Soft Cap (no exponentiation) ---
def compute_threat_score(df,
                         Cq=50, Cl=150, Cb=0,
                         Co=2,
                         baseline=100,
                         delta=150, theta=1,
                         score_mult=10,
                         conf_weight=0.5):
    df = df.copy()
    # ensure criticality term
    if 'S_C' not in df:
        df['S_C'] = Cq * df['Xcrit']**2 + Cl * df['Xcrit'] + Cb

    # observation & FP penalty
    df['S_O']        = Co * df['log_decayed_obs']
    df['FP_penalty'] = -delta * df['fp_count']**theta

    # raw + confidence
    df['raw_score'] = baseline + df['S_C'] + df['S_O'] + df['FP_penalty']
    df['raw_score'] *= (1 + conf_weight * df.get('conf_score', 0))

    # linear scaling
    df['scaled_score'] = (df['raw_score'] * score_mult).clip(lower=0)

    # clamp into 0–1000
    df['threat_score'] = df['scaled_score'].clip(upper=1000) \
                             .round().astype(int)

    return df

# --- 9. Severity Labeling ---
def compute_severity_label(df):
    df = df.copy()
    bins = [0, 199, 500, 800, 1000]
    labels = ['Low', 'Medium', 'High', 'Critical']
    df['severity'] = pd.cut(df['threat_score'], bins=bins, labels=labels, include_lowest=True)
    return df

# --- 10. Orchestrator Pipeline ---
def run_threat_pipeline(df,
                        daily_counts_df,
                        source_sightings_df,
                        trust_map,
                        incidents_df=None,
                        cal_scores=None,
                        lof_params=None,
                        scoring_params=None):
    df = compute_temporal_volume_features(df, daily_counts_df)
    df = compute_source_diversity(df, source_sightings_df, trust_map)
    df = compute_fp_signals(df)
    df = compute_context_metadata(df)
    df = compute_behavioral(df)
    df = compute_derived(df, incidents_df, cal_scores)
    df['Xcrit'] = (((df['rating'].clip(0,5)/5*2 - 2) + (df['confidence']/100*2 - 2)) / 2)
    feature_cols = [
        'S_C', 'log_decayed_obs', 'obs_trend_slope', 'obs_burstiness',
        'num_distinct_sources', 'sum_source_weight', 'fp_rate',
        'port_diversity', 'vt_malicious_count', 'port_scan_confidence',
        'peer_consensus_score'
    ]
        # Compute criticality term for confidence features
    df['S_C'] = 50 * df['Xcrit']**2 + 150 * df['Xcrit'] + 0
    # Estimate unsupervised confidence
    df = compute_confidence_score(df, feature_cols, lof_params)
    df = compute_threat_score(df, **(scoring_params or {}))
    df = compute_severity_label(df)
    return df

# --- 11. Main Execution ---
if __name__ == '__main__':
    # Load your full dataset
    df_full = recent_tags
    # Load or simulate auxiliary tables
    today = pd.Timestamp(datetime.utcnow().date())
    dates = [today - pd.Timedelta(days=i) for i in range(7)]
    daily = [{'indicator': ind, 'date': d, 'count': np.random.poisson(5)}
             for ind in df_full['indicator'] for d in dates]
    daily_counts_df = pd.DataFrame(daily)
    source_sightings_df = pd.DataFrame([
        {'indicator': ind, 'ownerId': oid}
        for ind in df_full['indicator'] for oid in df_full['ownerId'].unique()
    ])
    trust_map = {oid: np.random.choice([1, 5, 7, 10])
                 for oid in df_full['ownerId'].unique()}
    incidents_df, cal_scores = None, None
    # Run pipeline
    df_result = run_threat_pipeline(
        df_full,
        daily_counts_df,
        source_sightings_df,
        trust_map,
        incidents_df,
        cal_scores
    )
    # Display results
    display(df_result[['indicator', 'conf_score', 'threat_score', 'severity']])


C:\Users\jaskew\AppData\Local\Temp\ipykernel_23000\728928968.py:177: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = pd.Timestamp(datetime.utcnow().date())
C:\Users\jaskew\AppData\Local\Temp\ipykernel_23000\728928968.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = pd.Timestamp(datetime.utcnow().date())
C:\Users\jaskew\AppData\Local\Temp\ipykernel_23000\728928968.py:74: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['port_scan_confidence'

,indicator,conf_score,threat_score,severity
0,101.89.174.236,0.021199,312,Medium
1,103.124.139.170,0.076086,394,Medium
2,103.149.86.208,0.063640,378,Medium
3,103.186.30.186,0.069431,1000,Critical
4,103.203.59.0,0.571994,438,Medium
...,...,...,...,...
872,scan.cypex.ai,0.110182,368,Medium
873,svgarchive.com,0.014271,0,Low
874,vtext.com,0.012928,38,Low
875,www.sthda.com,0.203610,215,Medium


In [130]:
import pandas as pd

def scoring_breakdown(df):
    cols = [
        'indicator', 'rating', 'confidence', 'Xcrit', 'S_C', 'log_decayed_obs', 'S_O',
        'fp_count', 'FP_penalty', 'raw_score', 'scaled_score', 'threat_score', 'severity'
    ]
    breakdown = df[cols].copy()
    breakdown = breakdown.rename(columns={
        'S_C': 'Criticality Term',
        'S_O': 'Observation Term',
        'FP_penalty': 'False Positive Penalty',
        'raw_score': 'Raw Score',
        'scaled_score': 'Scaled Score',
        'exp_score': 'Exponentiated Score',
        'threat_score': 'Final Threat Score',
        'log_decayed_obs': 'Log Decayed Observations',
        'Xcrit': 'Criticality Index'
    })
    return breakdown

# Example usage:
score_details = scoring_breakdown(df_result)
display(score_details)

,indicator,rating,confidence,Criticality Index,Criticality Term,Log Decayed Observations,Observation Term,fp_count,False Positive Penalty,Raw Score,Scaled Score,Final Threat Score,severity
0,101.89.174.236,3.0,71.0,-0.69,-79.695,5.300928e+00,1.060186e+01,0,0,31.234451,312.344505,312,Medium
1,103.124.139.170,4.0,70.0,-0.50,-62.500,2.430372e-01,4.860745e-01,0,0,39.431182,394.311817,394,Medium
2,103.149.86.208,3.0,73.0,-0.67,-78.055,7.337883e+00,1.467577e+01,0,0,37.786048,377.860477,378,Medium
3,103.186.30.186,5.0,89.0,-0.11,-15.895,8.429855e+00,1.685971e+01,0,0,104.469747,1044.697470,1000,Critical
4,103.203.59.0,3.0,70.0,-0.70,-80.500,7.275069e+00,1.455014e+01,0,0,43.788369,437.883695,438,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...
872,scan.cypex.ai,3.0,77.0,-0.63,-74.655,4.753708e+00,9.507416e+00,0,0,36.772463,367.724628,368,Medium
873,svgarchive.com,0.0,0.0,-2.00,-100.000,5.658808e-06,1.131762e-05,0,0,0.000011,0.000114,0,Low
874,vtext.com,3.0,47.0,-0.93,-96.255,1.771958e-06,3.543915e-06,0,0,3.769210,37.692104,38,Low
875,www.sthda.com,4.0,50.0,-0.70,-80.500,5.312340e-17,1.062468e-16,0,0,21.485193,214.851930,215,Medium


In [131]:
score_details.head()

,indicator,rating,confidence,Criticality Index,Criticality Term,Log Decayed Observations,Observation Term,fp_count,False Positive Penalty,Raw Score,Scaled Score,Final Threat Score,severity
0,101.89.174.236,3.0,71.0,-0.69,-79.695,5.300928,10.601855,0,0,31.234451,312.344505,312,Medium
1,103.124.139.170,4.0,70.0,-0.50,-62.500,0.243037,0.486074,0,0,39.431182,394.311817,394,Medium
2,103.149.86.208,3.0,73.0,-0.67,-78.055,7.337883,14.675766,0,0,37.786048,377.860477,378,Medium
3,103.186.30.186,5.0,89.0,-0.11,-15.895,8.429855,16.859710,0,0,104.469747,1044.697470,1000,Critical
4,103.203.59.0,3.0,70.0,-0.70,-80.500,7.275069,14.550137,0,0,43.788369,437.883695,438,Medium


In [134]:
score_details[score_details['indicator'] == '207.167.64.24']

,indicator,rating,confidence,Criticality Index,Criticality Term,Log Decayed Observations,Observation Term,fp_count,False Positive Penalty,Raw Score,Scaled Score,Final Threat Score,severity
429,207.167.64.24,3.0,76.0,-0.64,-75.52,8.522813,17.045625,0,0,53.235874,532.358736,532,High
